# Exploratory Data Analysis

Load the processed COPD and retail ESD availability dataset.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import seaborn as sns

current_dir = Path.cwd().resolve()
search_dirs = [current_dir, *current_dir.parents]
project_dir = next((path for path in search_dirs if path.name == "disease_data"), None)

if project_dir is None:
    project_dir = next((path / "disease_data" for path in search_dirs if (path / "disease_data").exists()), None)

if project_dir is None:
    project_dir = next(
        (
            path / "26-the-data-miners-analysis" / "disease_data"
            for path in search_dirs
            if (path / "26-the-data-miners-analysis" / "disease_data").exists()
        ),
        None,
    )

if project_dir is None:
    raise FileNotFoundError("Could not find the disease_data project folder.")

processed_file = project_dir / "data" / "processed" / "ca_county_copd_retail_esd_merged.csv"

merged_df = pd.read_csv(processed_file, dtype={"county_fips": "string"})
merged_df.head()

## California County Choropleth Maps

Create county-level choropleth maps from the processed COPD and retail ESD availability dataset.

In [ ]:
import json

import geopandas as gpd

county_geojson_url = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
county_geojson_file = project_dir / "data" / "processed" / "ca_counties.geojson"

if county_geojson_file.exists():
    ca_counties = gpd.read_file(county_geojson_file)
else:
    all_counties = gpd.read_file(county_geojson_url)
    ca_counties = all_counties[all_counties["STATE"] == "06"].copy()
    ca_counties["fips"] = ca_counties["STATE"].astype(str).str.zfill(2) + ca_counties["COUNTY"].astype(str).str.zfill(3)
    ca_counties.to_file(county_geojson_file, driver="GeoJSON")

if "fips" not in ca_counties.columns:
    ca_counties["fips"] = ca_counties["STATE"].astype(str).str.zfill(2) + ca_counties["COUNTY"].astype(str).str.zfill(3)

ca_counties["fips"] = ca_counties["fips"].astype(str).str.zfill(5)
ca_geojson = json.loads(ca_counties.to_json())

for feature in ca_geojson["features"]:
    feature["id"] = feature["properties"]["fips"]

plot_df = merged_df.copy()
plot_df["county_fips"] = plot_df["county_fips"].astype("string").str.zfill(5)

figures_dir = project_dir / "results" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

def make_county_choropleth(value_column, title, color_label, color_scale):
    fig = px.choropleth(
        plot_df,
        geojson=ca_geojson,
        locations="county_fips",
        featureidkey="id",
        color=value_column,
        hover_name="county",
        hover_data={
            "county_fips": False,
            "retail_esd_availability_pct": ":.2f",
            "copd_adjusted_prevalence_pct": ":.1f",
        },
        color_continuous_scale=color_scale,
        labels={value_column: color_label},
    )
    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(
        title=title,
        height=600,
        margin={"r": 0, "t": 60, "l": 0, "b": 0},
        coloraxis_colorbar={"title": color_label},
    )
    return fig

### Figure 1. Retail ESD Availability by County

In [ ]:
retail_esd_fig = make_county_choropleth(
    value_column="retail_esd_availability_pct",
    title="Retail ESD Availability by California County",
    color_label="Retail ESD availability (%)",
    color_scale="Blues",
)

retail_esd_fig.write_html(figures_dir / "retail-esd-availability-choropleth.html", include_plotlyjs="cdn")
retail_esd_fig

### Figure 2. COPD Adjusted Prevalence by County

In [ ]:
copd_fig = make_county_choropleth(
    value_column="copd_adjusted_prevalence_pct",
    title="COPD Adjusted Prevalence by California County",
    color_label="COPD adjusted prevalence (%)",
    color_scale="Reds",
)

copd_fig.write_html(figures_dir / "copd-adjusted-prevalence-choropleth.html", include_plotlyjs="cdn")
copd_fig

### Retail ESD Availability vs. COPD Adjusted Prevalence

In [ ]:
scatter_fig = px.scatter(
    merged_df,
    x="retail_esd_availability_pct",
    y="copd_adjusted_prevalence_pct",
    hover_name="county",
    labels={
        "retail_esd_availability_pct": "Retail ESD availability (%)",
        "copd_adjusted_prevalence_pct": "COPD adjusted prevalence (%)",
    },
    title="Retail ESD Availability vs. COPD Adjusted Prevalence",
)
scatter_fig.show()
merged_df.describe()